# Attention Head Analysis

# Attention Head Analysis

## What This Is
Individual attention heads in transformers perform specific functions. Mech interp research has identified:

- **Induction heads**: Copy previous tokens — if the sequence has ...A B ... A, the head attends from the second A back to B, predicting B will follow again. Induction heads enable in-context learning.
- **Previous token heads**: Attend to the immediately preceding position
- **Duplicate token heads**: Detect repeated tokens in the sequence
- **Name movers**: Move information about named entities

Head ablation (zeroing out a head's output) reveals its functional importance. Activation patching identifies causal role.

In [ ]:
import numpy as np

np.random.seed(42)

# Analyze attention patterns for specific head behaviors

def create_induction_pattern(seq_len: int) -> np.ndarray:
    """Create an ideal induction head attention pattern."""
    # Attends from position i to position i-1 when the current token
    # matches a previous token
    pattern = np.eye(seq_len, k=-1)  # attend to previous position
    return pattern / (pattern.sum(axis=-1, keepdims=True) + 1e-10)

def create_previous_token_pattern(seq_len: int) -> np.ndarray:
    """Create a previous-token head pattern."""
    pattern = np.eye(seq_len, k=-1)
    return pattern / (pattern.sum(axis=-1, keepdims=True) + 1e-10)

def attention_head_similarity(attn_pattern: np.ndarray, template: np.ndarray) -> float:
    """Measure how similar an attention pattern is to a canonical template."""
    a_flat = attn_pattern.flatten()
    t_flat = template.flatten()
    if a_flat.std() < 1e-8 or t_flat.std() < 1e-8: return 0.0
    return float(np.corrcoef(a_flat, t_flat)[0, 1])

# Build a tiny transformer with some structure
D_MODEL = 16
VOCAB = 30
SEQ = 8
N_HEADS = 4
D_HEAD = 4

# Simulate attention patterns from different heads
def softmax(x, axis=-1):
    ex = np.exp(x - x.max(axis=axis, keepdims=True))
    return ex / ex.sum(axis=axis, keepdims=True)

def compute_attention(Q, K, mask=True):
    T = Q.shape[0]
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    if mask:
        scores += np.triu(np.ones((T,T)) * -1e9, k=1)
    return softmax(scores)

tokens = list(range(SEQ))
# Embed tokens
W_emb = np.random.randn(VOCAB, D_MODEL) * 0.2
X = W_emb[tokens]  # SEQ x D_MODEL

# Different head specializations
head_patterns = {}
for h in range(N_HEADS):
    if h == 0:  # Approximate induction head: Q attends to current, K to previous
        WQ = np.random.randn(D_MODEL, D_HEAD) * 0.3
        WK = np.roll(np.eye(D_MODEL, D_HEAD) * 0.5 + np.random.randn(D_MODEL, D_HEAD) * 0.1, -1, axis=0)
    elif h == 1:  # Previous token head: attends 1 step back
        WQ = np.random.randn(D_MODEL, D_HEAD) * 0.1
        WK = np.eye(D_MODEL, D_HEAD) * 0.5
    else:  # Generic heads
        WQ = np.random.randn(D_MODEL, D_HEAD) * 0.2
        WK = np.random.randn(D_MODEL, D_HEAD) * 0.2
    
    Q = X @ WQ; K = X @ WK
    head_patterns[h] = compute_attention(Q, K)

# Classify heads by attention pattern
templates = {
    'induction': create_induction_pattern(SEQ),
    'previous_token': create_previous_token_pattern(SEQ),
    'uniform': np.ones((SEQ, SEQ)) / SEQ,
}

print('Attention Head Analysis:')
print(f'{'Head':<6} {'Induction':>10} {'Prev Token':>11} {'Uniform':>9} {'Likely Role':<20}')
print('-' * 60)
for h in range(N_HEADS):
    sims = {t: attention_head_similarity(head_patterns[h], templates[t]) for t in templates}
    role = max(sims, key=sims.get)
    print(f'  H{h}   {sims["induction"]:>10.3f} {sims["previous_token"]:>11.3f} {sims["uniform"]:>9.3f} {role:<20}')
